In [0]:
import pandas as pd

query = """
SELECT d.date, f.aqi, f.pm25, f.pm10
FROM workspace.aqi.fact_aqi_daily f
JOIN workspace.aqi.dim_city c ON f.city_id = c.city_id
JOIN workspace.aqi.dim_date d ON f.date_id = d.date_id
WHERE c.city = 'Delhi'
ORDER BY d.date
"""

delhi = spark.sql(query).toPandas()
delhi["date"] = pd.to_datetime(delhi["date"])
delhi = delhi.set_index("date").asfreq("D")

print("Days:", len(delhi))
print("From", delhi.index.min().date(), "to", delhi.index.max().date())
delhi.head()

Days: 2009
From 2015-01-01 to 2020-07-01


,aqi,pm25,pm10
date,,,
2015-01-01,472.0,313.22,607.98
2015-01-02,454.0,186.18,269.55
2015-01-03,143.0,87.18,131.90
2015-01-04,319.0,151.84,241.84
2015-01-05,325.0,146.60,219.13


In [0]:
data = delhi.copy()

#Target: what we want to predict(tomorrow's AQI)
data["target_aqi_tomorrow"] = data["aqi"].shift(-1)

#Features: only information available today or earlier
data["aqi_today"] = data["aqi"]
data["aqi_lag1"] = data["aqi"].shift(1)
data["aqi_lag2"] = data["aqi"].shift(2)
data["aqi_lag7"] = data["aqi"].shift(7)
data["aqi_roll7"] = data["aqi"].rolling(7, min_periods = 5).mean()
data["aqi_roll7_std"] = data["aqi"].rolling(7, min_periods = 5).std()
data["pm25_today"] = data["pm25"]
data["pm10_today"] = data["pm10"]
data["month"] = data.index.month
data["day_of_year"] = data.index.dayofyear

feature_cols = ["aqi_today", "aqi_lag1", "aqi_lag2", "aqi_lag7", "aqi_roll7", "aqi_roll7_std", "pm25_today", "pm10_today", "month", "day_of_year"]

model_data = data.dropna(subset = feature_cols[:6] + ["target_aqi_tomorrow"])
print("Rows available for modelling:", len(model_data))
model_data[feature_cols + ["target_aqi_tomorrow"]].head()

Rows available for modelling: 1969


,aqi_today,aqi_lag1,aqi_lag2,aqi_lag7,aqi_roll7,aqi_roll7_std,pm25_today,pm10_today,month,day_of_year,target_aqi_tomorrow
date,,,,,,,,,,,
2015-01-08,383.0,353.0,318.0,472.0,327.857143,94.816364,229.90,360.95,1,8,375.0
2015-01-09,375.0,383.0,353.0,454.0,316.571429,80.993533,201.66,397.43,1,9,376.0
2015-01-10,376.0,375.0,383.0,143.0,349.857143,28.893071,221.02,361.74,1,10,379.0
2015-01-11,379.0,376.0,375.0,319.0,358.428571,27.054618,205.41,393.20,1,11,375.0
2015-01-12,375.0,379.0,376.0,325.0,365.571429,23.064093,212.41,345.63,1,12,366.0


In [0]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

train = model_data[model_data.index < "2019-01-01"]
test = model_data[(model_data.index >= '2019-01-01') & (model_data.index < '2020-01-01')]

print("Train rows:", len(train), "| from", train.index.min().date(), "to", train.index.max().date())
print("Test rows:", len(test), "| from", test.index.min().date(), "to", test.index.max().date())

def evaluate(name, actual, predicted):
    mae = mean_absolute_error(actual, predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    return {'model': name, "MAE": round(mae, 1), "RMSE": round(rmse, 1)}

y_test = test["target_aqi_tomorrow"]

results = [
    evaluate("Baseline: tomorrow = today", y_test, test['aqi_today']),
    evaluate("Baseline: tomorrow = 7-day average", y_test, test['aqi_roll7']),
]

pd.DataFrame(results)

Train rows: 1422 | from 2015-01-08 to 2018-12-31
Test rows: 365 | from 2019-01-01 to 2019-12-31


,model,MAE,RMSE
0,Baseline: tomorrow = today,38.6,54.1
1,Baseline: tomorrow = 7-day average,56.2,75.9


In [0]:
%pip install xgboost

Looking in indexes: [REDACTED]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.2/57.2 MB 158.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 305.1/305.1 MB 171.6 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [xgboost]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
from xgboost import XGBRegressor

X_train = train[feature_cols]
y_train = train["target_aqi_tomorrow"]
X_test = test[feature_cols]

model = XGBRegressor(
    n_estimators = 300,
    max_depth = 4,
    learning_rate = 0.05,
    subsample = 0.8,
    colsample_bytree = 0.8,
    random_state = 42 
)
model.fit(X_train, y_train)

test_pred = model.predict(X_test)
results.append(evaluate("XGBoost", y_test, test_pred))

pd.DataFrame(results)

,model,MAE,RMSE
0,Baseline: tomorrow = today,38.6,54.1
1,Baseline: tomorrow = 7-day average,56.2,75.9
2,XGBoost,32.1,44.2


In [0]:
%pip install shap

Looking in indexes: [REDACTED]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 547.3/547.3 kB 13.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [shap]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import shap

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

importance = (
    pd.DataFrame({
        "feature": feature_cols,
        "mean_abs_shap": np.abs(shap_values).mean(axis = 0),
    })
    .sort_values("mean_abs_shap", ascending = False)
    .round(1)
    .reset_index(drop = True)
)

importance

,feature,mean_abs_shap
0,pm25_today,46.900002
1,aqi_today,34.299999
2,pm10_today,15.100000
3,aqi_roll7,7.800000
4,aqi_lag2,3.700000
5,day_of_year,3.600000
6,aqi_lag1,3.500000
7,aqi_roll7_std,3.000000
8,aqi_lag7,3.000000
9,month,1.200000


In [0]:
threshold = 200

actual_poor = y_test > threshold
baseline_poor = test["aqi_today"] > threshold
xgb_poor = pd.Series(test_pred, index = test.index) > threshold


def warning_scores(name, actual, predicted):
    caught = (actual & predicted).sum()
    missed = (actual & ~predicted).sum()
    false_alarm = (~actual & predicted).sum()
    return {
        "model": name,
        "poor_days": int(actual.sum()),
        "caught": int(caught),
        "missed": int(missed),
        "false_alarms": int(false_alarm),
        "recall_%": round(caught / actual.sum() * 100, 1),
        "precision_%": round(caught / predicted.sum() * 100, 1)
    }
pd.DataFrame([
    warning_scores("Baseline: tomorrow = today", actual_poor, baseline_poor),
    warning_scores("XGBoost", actual_poor, xgb_poor)
])

,model,poor_days,caught,missed,false_alarms,recall_%,precision_%
0,Baseline: tomorrow = today,194,164,30,30,84.5,84.5
1,XGBoost,194,187,7,39,96.4,82.7


In [0]:
predictions = pd.DataFrame({
    "date": test.index,
    "city": "Delhi", 
    "actual_aqi_tomorrow": y_test.values,
    "baseline_pred": test["aqi_today"].values,
    "xgb_pred": np.round(test_pred, 1),
})

spark.createDataFrame(predictions).write.mode("overwrite").saveAsTable("workspace.aqi.gold_forecast_delhi_2019")

print("Saved", len(predictions), "predictions")

Saved 365 predictions
